# ResNet50: Model Evasion

We're going to do an image misclassification attack.  In this case we will be attacking the pretrained PyTorch [ResNet50 model](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html). This is an "open box" attack. Model inference code is provided below, including preprocessing code. You must reclassify the image from that of the actual class to a groom (class 982).  These are ImageNet labels. You can see a list [here](https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/).



In [ ]:
# DO NOT CHANGE
# Import the required libraries

from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms

import torch
import numpy as np

import requests
from io import BytesIO
from matplotlib import pyplot as plt
from PIL import Image

device = 'cuda'

In [ ]:
# Download and show the image
img_url = "https://raw.githubusercontent.com/SivadineshPonrajan/Adversarial-Machine-Learning-Workshop/main/assets/jamesbond.png"

response = requests.get(img_url, timeout=30)
response.raise_for_status()

img = Image.open(BytesIO(response.content)).convert("RGB")

In [ ]:
# move the imported model to the device, set eval mode, and load labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(device)
model.eval()
# Load labels from GitHub using requests

labels_url = "https://raw.githubusercontent.com/SivadineshPonrajan/Adversarial-Machine-Learning-Workshop/main/assets/labels.txt"

response = requests.get(labels_url, timeout=30)
response.raise_for_status()

labels = response.text.splitlines()

In [ ]:

preprocess = transforms.Compose([
    transforms.Resize(256),  # Resize the image to 256x256
    transforms.CenterCrop(224),  # Crop the image to 224x224 about the center
    transforms.ToTensor(),  # Convert the image to a PyTorch tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # Normalize the image with the ImageNet dataset mean values
        std=[0.229, 0.224, 0.225]  # Normalize the image with the ImageNet dataset standard deviation values
    )
]);

unnormalize = transforms.Normalize(
   mean= [-m/s for m, s in zip([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])],
   std= [1/s for s in [0.229, 0.224, 0.225]]
);

In [ ]:
plt.imshow(img)

In [ ]:
idx = model(preprocess(img).unsqueeze(0).to(device))[0].argmax()
print(idx, labels[idx])

In [ ]:
# YOUR CODE GOES HERE

# Find target class index for "laptop" (fallback to "notebook" if needed)
target_idx = None
for i, lab in enumerate(labels):
    if "groom" in lab.lower():
        target_idx = i
        break

print("Target index:", target_idx, "| label:", labels[target_idx] if target_idx is not None else None)

In [ ]:
# Work in pixel space [0,1] with a 256x256 image so the saved file preserves size
img_256 = img.resize((256, 256))
to_tensor = transforms.ToTensor()
to_pil = transforms.ToPILImage()

x0 = to_tensor(img_256).to(device)  # shape [3,256,256], values in [0,1]
x = x0.clone().detach().requires_grad_(True)

# Helper: normalize like the model expects (same stats as in preprocess)
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(3,1,1)

def normalize_pixel_tensor(t):
    return (t - mean) / std

In [ ]:
# Targeted iterative attack (PGD-like)
eps = 8/255        # max per-pixel change (keep small so rooster still looks like rooster)
alpha = 1/255      # step size
steps = 50         # more steps = stronger attack

target = torch.tensor([target_idx], device=device)

for step in range(steps):
    # forward
    logits = model(normalize_pixel_tensor(x).unsqueeze(0))

    # targeted: minimize CE toward target class
    loss = torch.nn.functional.cross_entropy(logits, target)

    # backward
    loss.backward()

    with torch.no_grad():
        # gradient descent step (because we're minimizing loss to target)
        x -= alpha * x.grad.sign()

        # project back into eps-ball around original image
        x = torch.max(torch.min(x, x0 + eps), x0 - eps)

        # clamp to valid pixel range
        x = x.clamp(0, 1)

    x = x.detach().requires_grad_(True)

In [ ]:
adv_img = to_pil(x.cpu())
# Check
idx_after = model(preprocess(adv_img).unsqueeze(0).to(device))[0].argmax().item()
print("After attack:", idx_after, labels[idx_after])